# Sub-Category: PGA-UNet vs U-Net vs Attention U-Net

| Section | Nội dung |
|---|---|
| **0** | U-Net chạy 187 → sort Dice → lấy **Easy top-100** + **Hard bottom-100** (danh sách mẫu) |
| **1** | U-Net chạy 200 mẫu → 6 metrics + visualize 10+10 |
| **2** | Att-UNet chạy 200 mẫu → 6 metrics + visualize 10+10 |
| **3** | PGA-UNet chạy 200 mẫu (per-polygon→avg/image) → 6 metrics + visualize 10+10 |
| **Agg** | Bảng tổng hợp 3 model × 2 nhóm + bar chart |

Visualization 4 cột: **Ảnh** | **GT** | **Dự đoán** *(Dice/IoU/CBL)* | **Overlap** *(Pre/Rec/HD95)*

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1 — SETUP (clone, dataset, checkpoints, utilities)
# ══════════════════════════════════════════════════════
%cd /content
import os, sys, cv2, json as _json
import numpy as np, torch
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle as Rect
!pip install -q gdown tqdm opencv-python matplotlib scipy
import gdown

REPO='https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git'
for d,b in [('unet-repo',None),('attunet-repo','attention-unet2D'),('pga-repo','TN_B_ON')]:
    if not os.path.exists(f'/content/{d}'):
        br=f'-b {b} ' if b else ''
        os.system(f'git clone -q {br}{REPO} /content/{d}')
    print(f'  ✅ {d}')

if not os.path.exists('/content/dataset_BTXRD'):
    gdown.download('https://drive.google.com/uc?id=1Yg97cO5W2pYKV30wJQCl24xOEKQCwunI',
                   '/content/dataset_BTXRD.zip', quiet=False)
    os.system('unzip -q /content/dataset_BTXRD.zip -d /content/')
for d in ['unet-repo','attunet-repo','pga-repo']:
    if not os.path.exists(f'/content/{d}/dataset_BTXRD'):
        os.system(f'cp -r /content/dataset_BTXRD /content/{d}/dataset_BTXRD')
print('  ✅ Dataset ready')

os.makedirs('/content/checkpoints',exist_ok=True)
for cid,fn in [('15529s6z9PgrL-Jy95ej5wILLPXcmJiKI','unet_best.pth'),
               ('1zL5xeZAyHQ-Uxifr3W12rfEr69qtfuMN','attunet_best.pth'),
               ('1Mv-rUPI7KGmYemd27hmKbJQRHc4ZKB9z', 'pga_unet_expB_best.pth')]:
    p=f'/content/checkpoints/{fn}'
    if not os.path.exists(p): gdown.download(f'https://drive.google.com/uc?id={cid}',p,quiet=False)
    print(f'  ✅ {fn}  {os.path.getsize(p)//1024}KB')

DEVICE,IMG_SIZE,ZOOM_R=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),512,0.30
IMG_DIR ='/content/unet-repo/dataset_BTXRD/test/images'
MASK_DIR='/content/unet-repo/dataset_BTXRD/test/masks'
JSON_DIR='/content/pga-repo/dataset_BTXRD/test/annotations'
KEYS=['dice','iou','precision','recall','hd95','cbl']

def extract_lcc(m):
    if m.sum()==0: return m
    n,lbl,st,_=cv2.connectedComponentsWithStats(m.astype(np.uint8),8)
    return m if n<=1 else (lbl==(1+np.argmax(st[1:,cv2.CC_STAT_AREA]))).astype(np.float32)

def calc_metrics(prob,gt,eps=1e-6):
    pm=extract_lcc((prob>0.5).astype(np.float32)); gm=(gt>0.5).astype(np.float32)
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    p,g=pm.astype(bool),gm.astype(bool); hd95=float(IMG_SIZE)
    if p.any() and g.any():
        pe=p^binary_erosion(p); ge=g^binary_erosion(g)
        d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
        if len(d1) and len(d2): hd95=float(max(np.percentile(d1,95),np.percentile(d2,95)))
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)),iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95,cbl=cbl,mask=pm)

def plateau_heatmap(bbox,S=512):
    x1=max(0,int(bbox[0])-5);y1=max(0,int(bbox[1])-5)
    x2=min(S,int(bbox[2])+5);y2=min(S,int(bbox[3])+5)
    hm=np.zeros((S,S),dtype=np.float32)
    if x2>x1 and y2>y1: hm[y1:y2,x1:x2]=1.0; hm=cv2.GaussianBlur(hm,(31,31),0)
    return hm

def print_metrics(label,mlist):
    m={k:np.mean([r[k] for r in mlist]) for k in KEYS}
    print(f"  {label:<16} Dice={m['dice']:.4f} IoU={m['iou']:.4f} Pre={m['precision']:.4f}"
          f" Rec={m['recall']:.4f} HD95={m['hd95']:.2f} CBL={m['cbl']:.4f}")
    return m

def visualize(records, title, n=10, show_bbox=False):
    recs=records[:n]; nr=len(recs)
    fig,axes=plt.subplots(nr,4,figsize=(20,4.5*nr),squeeze=False)
    fig.suptitle(title,fontsize=13,fontweight='bold',y=1.005)
    for j,t in enumerate(['Ảnh gốc','Ground Truth',
                           'Dự đoán\n(Dice / IoU / CBL)',
                           'Overlap\n(Pre / Rec / HD95)']):
        axes[0,j].set_title(t,fontsize=9,fontweight='bold')
    for row,rec in enumerate(recs):
        gray=np.clip((rec['img_np']*0.5+0.5)*255,0,255).astype(np.uint8)
        rgb=cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)
        gt=rec['gt']; pm=rec['m']['mask']
        def ov(mask,clr,a=0.5):
            o=rgb.copy().astype(np.float32); o[mask>0.5]=o[mask>0.5]*(1-a)+np.array(clr)*a
            return np.clip(o,0,255).astype(np.uint8)
        diff=rgb.copy(); pb,gb=pm>0.5,gt>0.5
        diff[gb&~pb]=[30,180,30]; diff[pb&~gb]=[220,60,60]; diff[pb&gb]=[220,200,0]
        ax=axes[row,0]; ax.imshow(gray,cmap='gray')
        if show_bbox:
            for bx in rec.get('bboxes',[]):
                ax.add_patch(Rect((bx[0],bx[1]),bx[2]-bx[0],bx[3]-bx[1],lw=1.5,edgecolor='lime',facecolor='none'))
        ax.set_ylabel(f"#{rec['idx']}",fontsize=8,rotation=0,labelpad=30,va='center'); ax.axis('off')
        axes[row,1].imshow(ov(gt,[30,120,255])); axes[row,1].axis('off')
        m=rec['m']
        axes[row,2].imshow(ov(pm,[220,60,60]))
        axes[row,2].set_title(f"Dice={m['dice']:.3f}  IoU={m['iou']:.3f}  CBL={m['cbl']:.3f}",
                              fontsize=8,color='darkred',pad=2); axes[row,2].axis('off')
        axes[row,3].imshow(diff)
        axes[row,3].set_title(f"Pre={m['precision']:.3f}  Rec={m['recall']:.3f}  HD95={m['hd95']:.1f}px",
                              fontsize=8,color='saddlebrown',pad=2); axes[row,3].axis('off')
    plt.tight_layout()
    fname=title.split('|')[0].strip()[:6].lower()+'_'+title.split('|')[-1].strip()[:4].lower()+'.png'
    plt.savefig(fname,dpi=100,bbox_inches='tight'); plt.show(); print(f'  ✅ {fname}')

# Load dataset & image stems
sys.path.insert(0,'/content/unet-repo')
from dataset import BTXRD_Dataset
test_ds=BTXRD_Dataset(image_dir=IMG_DIR,mask_dir=MASK_DIR,img_size=IMG_SIZE,is_train=False)
img_stems=sorted([os.path.splitext(f)[0] for f in os.listdir(IMG_DIR)
                  if f.lower().endswith(('.png','.jpg','.jpeg'))])
all_raw=[]
for idx,(img_t,mask_t) in enumerate(DataLoader(test_ds,batch_size=1,shuffle=False)):
    # .copy() để tránh DataLoader tái sử dụng memory buffer
    all_raw.append({'idx':idx,
                    'img_np':img_t[0,0].numpy().copy(),
                    'gt':mask_t[0,0].numpy().copy(),
                    'img_t':img_t.clone(),
                    'stem':img_stems[idx]})
easy_idx=[]; hard_idx=[]
SEC={} # lưu metrics mỗi section: SEC['unet']={'easy':[...], 'hard':[...]}
print(f'\n✅ Setup xong | {len(all_raw)} samples | device={DEVICE}')


In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 0 — Phân nhóm bằng U-Net (chạy 187 → sort)
# ══════════════════════════════════════════════════════

%cd /content/unet-repo
if '/content/unet-repo' not in sys.path:
    sys.path.insert(0, '/content/unet-repo')
else:
    sys.path.remove('/content/unet-repo'); sys.path.insert(0, '/content/unet-repo')

from models.networks.unet_2D import unet_2D
_unet_cat=unet_2D(in_channels=1,n_classes=1).to(DEVICE)
_unet_cat.load_state_dict(torch.load('/content/checkpoints/unet_best.pth',map_location=DEVICE,weights_only=True))
_unet_cat.eval(); print('✅ U-Net loaded (categorization)')

dice_all=[]
with torch.no_grad():
    for s in tqdm(all_raw,'Sec0-UNet'):
        prob=torch.sigmoid(_unet_cat(s['img_t'].to(DEVICE)))[0,0].cpu().numpy()
        dice_all.append(calc_metrics(prob,s['gt'])['dice'])

sorted_desc=sorted(range(len(all_raw)),key=lambda i:dice_all[i],reverse=True)
easy_idx=sorted_desc[:100]    # top-100 UNet Dice cao nhất → Easy
hard_idx =sorted_desc[-100:]  # bottom-100 UNet Dice thấp nhất → Hard
del _unet_cat  # giải phóng bộ nhớ

print(f'\n  Easy top-100  Dice range: {dice_all[easy_idx[0]]:.4f} → {dice_all[easy_idx[-1]]:.4f}')
print(f'  Hard bot-100  Dice range: {dice_all[hard_idx[0]]:.4f} → {dice_all[hard_idx[-1]]:.4f}')
print(f'  Tổng mẫu đánh giá: {len(set(easy_idx+hard_idx))} ảnh khác nhau (có thể overlap trung gian)')


In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 1 — U-Net 2D  (branch: main)
# Chạy 200 mẫu (100 Easy + 100 Hard)
# ══════════════════════════════════════════════════════

%cd /content/unet-repo
# Clear module cache để import từ repo khác
for _k in list(sys.modules.keys()):
    if 'models' in _k: del sys.modules[_k]
if '/content/unet-repo' not in sys.path:
    sys.path.insert(0, '/content/unet-repo')
else:
    sys.path.remove('/content/unet-repo'); sys.path.insert(0, '/content/unet-repo')

from models.networks.unet_2D import unet_2D
unet=unet_2D(in_channels=1,n_classes=1).to(DEVICE)
unet.load_state_dict(torch.load('/content/checkpoints/unet_best.pth',map_location=DEVICE,weights_only=True))
unet.eval(); print('✅ U-Net loaded')

res_u_easy,res_u_hard=[],[]
with torch.no_grad():
    for i in tqdm(easy_idx,'UNet-Easy'):
        prob=torch.sigmoid(unet(all_raw[i]['img_t'].to(DEVICE)))[0,0].cpu().numpy()
        res_u_easy.append(calc_metrics(prob,all_raw[i]['gt']))
    for i in tqdm(hard_idx,'UNet-Hard'):
        prob=torch.sigmoid(unet(all_raw[i]['img_t'].to(DEVICE)))[0,0].cpu().numpy()
        res_u_hard.append(calc_metrics(prob,all_raw[i]['gt']))

bar='='*70; print(f'\n{bar}\n  SECTION 1 — U-Net\n{bar}')
SEC['unet']={'easy':print_metrics('Easy (N=100)',res_u_easy),
             'hard':print_metrics('Hard (N=100)',res_u_hard)}

recs_e=[dict(**all_raw[easy_idx[k]],m=res_u_easy[k]) for k in range(len(easy_idx))]
recs_h=[dict(**all_raw[hard_idx[k]],m=res_u_hard[k]) for k in range(len(hard_idx))]
visualize(recs_e,'U-Net | EASY — 10 ảnh',n=10)
visualize(recs_h[-10:][::-1],'U-Net | HARD — 10 ảnh tệ nhất (từ dưới lên)',n=10)


In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 2 — Attention U-Net 2D  (branch: attention-unet2D)
# Chạy 200 mẫu (100 Easy + 100 Hard)
# ══════════════════════════════════════════════════════

%cd /content/attunet-repo
# Clear module cache để import từ repo khác
for _k in list(sys.modules.keys()):
    if 'models' in _k: del sys.modules[_k]
if '/content/attunet-repo' not in sys.path:
    sys.path.insert(0, '/content/attunet-repo')
else:
    sys.path.remove('/content/attunet-repo'); sys.path.insert(0, '/content/attunet-repo')

from models.networks.attention_unet_2D import Attention_UNet_2D
attunet=Attention_UNet_2D(in_channels=1,n_classes=1).to(DEVICE)
attunet.load_state_dict(torch.load('/content/checkpoints/attunet_best.pth',map_location=DEVICE,weights_only=True))
attunet.eval(); print('✅ Att-UNet loaded')

res_a_easy,res_a_hard=[],[]
with torch.no_grad():
    for i in tqdm(easy_idx,'AttUNet-Easy'):
        prob=torch.sigmoid(attunet(all_raw[i]['img_t'].to(DEVICE)))[0,0].cpu().numpy()
        res_a_easy.append(calc_metrics(prob,all_raw[i]['gt']))
    for i in tqdm(hard_idx,'AttUNet-Hard'):
        prob=torch.sigmoid(attunet(all_raw[i]['img_t'].to(DEVICE)))[0,0].cpu().numpy()
        res_a_hard.append(calc_metrics(prob,all_raw[i]['gt']))

bar='='*70; print(f'\n{bar}\n  SECTION 2 — Att-UNet\n{bar}')
SEC['attunet']={'easy':print_metrics('Easy (N=100)',res_a_easy),
                'hard':print_metrics('Hard (N=100)',res_a_hard)}

recs_e=[dict(**all_raw[easy_idx[k]],m=res_a_easy[k]) for k in range(len(easy_idx))]
recs_h=[dict(**all_raw[hard_idx[k]],m=res_a_hard[k]) for k in range(len(hard_idx))]
visualize(recs_e,'Att-UNet | EASY — 10 ảnh',n=10)
visualize(recs_h[-10:][::-1],'Att-UNet | HARD — 10 ảnh tệ nhất (từ dưới lên)',n=10)


In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 3 — PGA-UNet  (branch: TN_B_ON)
# Load ảnh trực tiếp từ disk cho mỗi sample (tránh tensor cache bug)
# Per-polygon: mỗi polygon 1 zoom-out bbox riêng
# ══════════════════════════════════════════════════════
%cd /content/pga-repo
for _k in list(sys.modules.keys()):
    if 'models' in _k: del sys.modules[_k]
if '/content/pga-repo' not in sys.path:
    sys.path.insert(0, '/content/pga-repo')
else:
    sys.path.remove('/content/pga-repo'); sys.path.insert(0, '/content/pga-repo')
from models.networks.prompt_unet_2D import PGA_UNet

pga = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
pga.load_state_dict(torch.load('/content/checkpoints/pga_unet_expB_best.pth',
                                map_location=DEVICE, weights_only=True))
pga.eval(); print('✅ PGA-UNet loaded')

JSON_DIR_PGA = '/content/pga-repo/dataset_BTXRD/test/annotations'

def load_fresh_image(stem):
    """Load ảnh trực tiếp từ disk → img_np (float32 norm) + img_t ([1,1,512,512])"""
    ip = next((f'{IMG_DIR}/{stem}{e}' for e in ('.png','.jpg','.jpeg')
                if os.path.exists(f'{IMG_DIR}/{stem}{e}')), None)
    if ip is None: return None, None
    bgr = cv2.imread(ip)
    gray = cv2.cvtColor(cv2.resize(bgr,(IMG_SIZE,IMG_SIZE)),cv2.COLOR_BGR2GRAY)
    img_np = (gray.astype(np.float32)/255. - 0.5) / 0.5
    img_t  = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).float()
    return img_np, img_t

def load_pga_polygon_samples(img_indices):
    """Với mỗi ảnh, load từ disk + tất cả polygon trong JSON
    → list of dicts: 1 dict = 1 polygon sample (img_np, img_t, gt, bbox đúng)
    """
    samples = []
    img_cache = {}  # stem → (img_np, img_t) để tránh đọc lại nhiều lần
    for img_i in img_indices:
        stem = all_raw[img_i]['stem']
        if stem not in img_cache:
            img_np, img_t = load_fresh_image(stem)
            if img_np is None: continue
            img_cache[stem] = (img_np, img_t)
        img_np, img_t = img_cache[stem]

        jp = os.path.join(JSON_DIR_PGA, stem + '.json')
        if not os.path.exists(jp):
            # Fallback: merged PNG mask bbox
            gt = all_raw[img_i]['gt']
            ys, xs = np.where(gt > 0.5)
            if not len(xs): continue
            bw=xs.max()-xs.min(); bh=ys.max()-ys.min()
            bbox=[max(0,xs.min()-bw*ZOOM_R), max(0,ys.min()-bh*ZOOM_R),
                  min(IMG_SIZE,xs.max()+bw*ZOOM_R), min(IMG_SIZE,ys.max()+bh*ZOOM_R)]
            samples.append(dict(img_i=img_i,stem=stem,sidx=0,
                               img_np=img_np,img_t=img_t,gt=gt.copy(),bbox=bbox))
            continue

        with open(jp,encoding='utf-8') as f: data=_json.load(f)
        iw=data.get('imageWidth',IMG_SIZE); ih=data.get('imageHeight',IMG_SIZE)
        sx,sy=IMG_SIZE/iw, IMG_SIZE/ih
        polys=[s for s in data.get('shapes',[]) if s.get('shape_type')=='polygon']
        for sidx,shp in enumerate(polys):
            pts=np.array(shp['points'],dtype=np.float32)*np.array([sx,sy])
            gt_mask=np.zeros((IMG_SIZE,IMG_SIZE),dtype=np.uint8)
            cv2.fillPoly(gt_mask,[pts.astype(np.int32)],255)
            gt_f=(gt_mask/255.).astype(np.float32)
            # Zoom-out bbox riêng cho từng polygon
            xs2,ys2=pts[:,0],pts[:,1]
            bw=xs2.max()-xs2.min(); bh=ys2.max()-ys2.min()
            bbox=[max(0,xs2.min()-bw*ZOOM_R), max(0,ys2.min()-bh*ZOOM_R),
                  min(IMG_SIZE,xs2.max()+bw*ZOOM_R), min(IMG_SIZE,ys2.max()+bh*ZOOM_R)]
            samples.append(dict(img_i=img_i,stem=stem,sidx=sidx,
                               img_np=img_np,img_t=img_t,gt=gt_f,bbox=bbox))
    return samples

# Load per-polygon samples
easy_pga = load_pga_polygon_samples(easy_idx)
hard_pga  = load_pga_polygon_samples(hard_idx)
print(f'  Easy: {len(easy_idx)} ảnh → {len(easy_pga)} polygon samples')
print(f'  Hard: {len(hard_idx)} ảnh → {len(hard_pga)} polygon samples')

# Inference per polygon với zoom-out bbox riêng
with torch.no_grad():
    for s in tqdm(easy_pga,'PGA-Easy'):
        hm=plateau_heatmap(s['bbox'])
        hm_t=torch.from_numpy(hm).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
        prob=torch.sigmoid(pga(s['img_t'].to(DEVICE),hm_t))[0,0].cpu().numpy()
        s['m']=calc_metrics(prob,s['gt'])
    for s in tqdm(hard_pga,'PGA-Hard'):
        hm=plateau_heatmap(s['bbox'])
        hm_t=torch.from_numpy(hm).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
        prob=torch.sigmoid(pga(s['img_t'].to(DEVICE),hm_t))[0,0].cpu().numpy()
        s['m']=calc_metrics(prob,s['gt'])

bar='='*70
print(f'\n{bar}\n  SECTION 3 — PGA-UNet (per-polygon, zoom-out 30%)\n{bar}')
def pm(label,lst): return print_metrics(label,[s['m'] for s in lst])
SEC['pga']={'easy':pm(f'Easy  ({len(easy_pga):3d} samples/{len(easy_idx)} ảnh)',easy_pga),
            'hard':pm(f'Hard  ({len(hard_pga):3d} samples/{len(hard_idx)} ảnh)',hard_pga),
            'n_easy':len(easy_pga),'n_hard':len(hard_pga)}

# Visualize: mỗi polygon = 1 hàng
# Easy: 10 ảnh tốt nhất | Hard: 10 ảnh tệ nhất
def pga_vis(pga_samples, vis_img_indices, title):
    vis_set=set(vis_img_indices)
    recs=[s for s in pga_samples if s['img_i'] in vis_set]
    if not recs: print('  (no samples)'); return
    nr=len(recs)
    fig,axes=plt.subplots(nr,4,figsize=(20,4.5*nr),squeeze=False)
    fig.suptitle(f'{title}  ({len(vis_img_indices)} ảnh / {nr} polygon samples)',
                 fontsize=13,fontweight='bold',y=1.005)
    for j,t in enumerate(['Ảnh + Prompt BBox','Ground Truth',
                           'Dự đoán\n(Dice / IoU / CBL)',
                           'Overlap\n(Pre / Rec / HD95)']):
        axes[0,j].set_title(t,fontsize=9,fontweight='bold')
    for row,s in enumerate(recs):
        gray=np.clip((s['img_np']*0.5+0.5)*255,0,255).astype(np.uint8)
        rgb=cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)
        gt=s['gt']; pm_mask=s['m']['mask']; bx=s['bbox']
        def ov(mask,clr,a=0.5):
            o=rgb.copy().astype(np.float32)
            o[mask>0.5]=o[mask>0.5]*(1-a)+np.array(clr)*a
            return np.clip(o,0,255).astype(np.uint8)
        diff=rgb.copy(); pb,gb=pm_mask>0.5,gt>0.5
        diff[gb&~pb]=[30,180,30]; diff[pb&~gb]=[220,60,60]; diff[pb&gb]=[220,200,0]
        # Col 0: ảnh + bbox đúng của polygon này
        ax=axes[row,0]; ax.imshow(gray,cmap='gray')
        ax.add_patch(Rect((bx[0],bx[1]),bx[2]-bx[0],bx[3]-bx[1],
                          lw=1.5,edgecolor='lime',facecolor='none'))
        ax.set_ylabel(f"{s['stem']}\npoly#{s['sidx']}",
                      fontsize=7,rotation=0,labelpad=75,va='center')
        ax.axis('off')
        # Col 1: GT của polygon này
        axes[row,1].imshow(ov(gt,[30,120,255])); axes[row,1].axis('off')
        # Col 2: dự đoán PGA
        m=s['m']
        axes[row,2].imshow(ov(pm_mask,[220,60,60]))
        axes[row,2].set_title(f"Dice={m['dice']:.3f}  IoU={m['iou']:.3f}  CBL={m['cbl']:.3f}",
                              fontsize=8,color='darkred',pad=2); axes[row,2].axis('off')
        # Col 3: overlap
        axes[row,3].imshow(diff)
        axes[row,3].set_title(f"Pre={m['precision']:.3f}  Rec={m['recall']:.3f}  HD95={m['hd95']:.1f}px",
                              fontsize=8,color='saddlebrown',pad=2); axes[row,3].axis('off')
    plt.tight_layout()
    fname=title.split('|')[1].strip()[:4].lower()+'_pga.png'
    plt.savefig(fname,dpi=100,bbox_inches='tight'); plt.show()
    print(f'  ✅ {fname}  ({nr} hàng = polygon samples)')

pga_vis(easy_pga, easy_idx[:10],    'PGA | EASY — 10 ảnh tốt nhất')
pga_vis(hard_pga, hard_idx[-10:],   'PGA | HARD — 10 ảnh tệ nhất')


In [ ]:
# ══════════════════════════════════════════════════════
# CELL TỔNG HỢP — So sánh 3 model × 2 nhóm
# ══════════════════════════════════════════════════════
import csv, os

bar='═'*82
print(f'\n{bar}')
print('  BẢNG TỔNG KẾT  |  Easy = top-100 UNet Dice  |  Hard = bottom-100 UNet Dice')
print(f'{bar}')
print(f"  {'Nhóm':<8} {'Model':<14} {'Dice':>7} {'IoU':>7} {'Pre':>7} {'Rec':>7} {'HD95':>8} {'CBL':>7}")
print(f"  {'─'*74}")

csv_rows=[]
for grp in ['easy','hard']:
    for model,key in [('U-Net','unet'),('AttUNet','attunet'),('PGA ←','pga')]:
        m=SEC[key][grp]
        print(f"  {grp.upper():<8} {model:<14}"
              f" {m['dice']:>7.4f} {m['iou']:>7.4f} {m['precision']:>7.4f}"
              f" {m['recall']:>7.4f} {m['hd95']:>8.2f} {m['cbl']:>7.4f}")
        n_val = SEC['pga'].get(f'n_{grp}',100) if key=='pga' else 100
        csv_rows.append([grp.upper(),model,n_val]+[f"{m[k]:.4f}" for k in KEYS])
    pu=SEC['unet'][grp]; pp=SEC['pga'][grp]
    print(f"  {'':8} {'Δ PGA − UNet':<14} {pp['dice']-pu['dice']:>+7.4f}")
    print(f"  {'─'*74}")
print(bar)

os.makedirs('results',exist_ok=True)
with open('results/subcat_pga_vs_baseline.csv','w',newline='') as f:
    csv.writer(f).writerows([['group','model','N']+KEYS]+csv_rows)
print('→ CSV: results/subcat_pga_vs_baseline.csv')

fig,ax=plt.subplots(figsize=(8,5))
x=np.arange(2); w=0.25
for i,(label,key,clr) in enumerate([('U-Net','unet','#6699cc'),('AttUNet','attunet','#66bb6a'),('PGA-UNet','pga','#ef5350')]):
    vals=[SEC[key]['easy']['dice'],SEC[key]['hard']['dice']]
    bars=ax.bar(x+(i-1)*w,vals,w,label=label,color=clr,edgecolor='white')
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2,v+0.012,f'{v:.3f}',ha='center',fontsize=9,fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(['EASY (N=100)','HARD (N=100)'],fontsize=11)
ax.set_ylabel('Dice'); ax.set_ylim(0,1.12)
ax.legend(fontsize=10); ax.grid(axis='y',alpha=0.3)
ax.set_title('PGA-UNet vs Baseline — Easy vs Hard',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('results/subcat_baseline_bar.png',dpi=130,bbox_inches='tight'); plt.show()
print('→ Plot: results/subcat_baseline_bar.png')
